# 09C RNNs/LSTMs vs Transformers (with Attention)

* In 09A we saw that RNNs (and by extension LSTMs) can learn time-series data to predict the next data points (and even a sequence of data points). 

* RNNs / LSTMs can also be trained effectively to predict the next word in a sequence, given enough training data. 

    * `"The cat sat on the ..."` (mat)
    * `"The quick brown fox jumps over the lazy ..." ` (dog)
    * `"I love machine ..."` (learning)

* However, we're going to test the limits of their 'long-term dependencies' by classifying statements, which will feature positive words, but are in fact sarcastic! 

## 0. Setup - Common Imports

`pip install scikit-learn`  
`python3 -m pip install -U scikit-learn --user`

`pip install tensorflow`  
`python3 -m pip install -U tensorflow --user`

`pip install torch`  
`python3 -m pip install -U torch --user`

`pip install transformers`  
`python3 -m pip install -U transformers --user`


In [120]:
from transformers import AutoTokenizer, BertTokenizer, BertModel

In [123]:
# Python ≥3.8 is required
import sys
assert sys.version_info >= (3, 8)

# Scikit-Learn ≥1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import seaborn as sns
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "rnn"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## 0. Setup - TensorFlow check

In [124]:
try:
    # %tensorflow_version only exists in Colab.
    %tensorflow_version 2.x
    IS_COLAB = True
except Exception:
    IS_COLAB = False
    
# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

tf.random.set_seed(42)

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print(tf.config.experimental.list_physical_devices('GPU'))

# Check for TensorFlow GPU access
print(f"TensorFlow has access to the following devices:\n{tf.config.list_physical_devices()}")
# See TensorFlow version
print(f"TensorFlow version: {tf.__version__}")


if not tf.test.is_gpu_available():
    print("No GPU was detected. LSTMs and RNNs can be very slow without a GPU.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")


Num GPUs Available:  0
[]
TensorFlow has access to the following devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
TensorFlow version: 2.18.0
No GPU was detected. LSTMs and RNNs can be very slow without a GPU.


## 0. Setup - PyTorch check

In [125]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Check if Metal Performance Shaders is available - on macOS
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal Performance Shaders) device.")
    
# Check for CUDA (NVIDIA GPU)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Properties: {torch.cuda.get_device_properties(0)}")
    
# Default to CPU if no accelerators are available
else:
    device = torch.device("cpu")
    print("Using CPU device.")

# Display summary of available devices
print(f"Selected device: {device}")
print("Available devices summary:")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"MPS Available: {torch.backends.mps.is_available()}")

Using MPS (Metal Performance Shaders) device.
Selected device: mps
Available devices summary:
CUDA Available: False
MPS Available: True


## 1. Small 'sarcastic' dataset - binary classification

* Now that we're set up, observe the small dataset below

* You'll see that it's a binary classification task, and we've provided the labels (so supervised learning activity)

In [ ]:
#import torch
#import torch.nn as nn
#import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

In [ ]:
sarcasm_data = [
    ("Oh great, another Monday at work!", 1),  # Sarcasm
    ("I love spending time with my family.", 0),  # Not Sarcasm
    ("Wow, what a fantastic traffic jam.", 1),
    ("This restaurant has amazing food!", 0),
    ("Just what I needed, more assignments.", 1)
]

I appreciate some of these could actually be read both ways... but for the purpose of our training, we'll see if our LSTMs can tell the difference.

In [189]:
sarcasm_data = [
    ("Oh great, another Monday at work!", 1),  # Sarcasm
    ("I love spending time with my family.", 0),  # Not Sarcasm
    ("Wow, what a fantastic traffic jam.", 1),  # Sarcasm
    ("This restaurant has amazing food!", 0),  # Not Sarcasm
    ("Just what I needed, more assignments.", 1),  # Sarcasm
    ("I’m so glad it’s raining today.", 1),  # Sarcasm
    ("The Wi-Fi is down again? That’s awesome.", 1),  # Sarcasm
    ("I can't wait to wake up early tomorrow.", 1),  # Sarcasm
    ("I absolutely love waiting in long lines.", 1),  # Sarcasm
    ("Finally, a nice quiet evening with no distractions.", 0),  # Not Sarcasm
    ("This project is going to be a breeze!", 1),  # Sarcasm
    ("I’m really excited to go to the dentist today.", 1),  # Sarcasm
    ("How wonderful, another email about a meeting.", 1),  # Sarcasm
    ("I just adore getting caught in traffic.", 1),  # Sarcasm
    ("I’m thrilled that my flight got delayed.", 1),  # Sarcasm
    ("What a lovely surprise to see you here.", 0),  # Not Sarcasm
    ("Oh, perfect timing for a power outage.", 1)  # Sarcasm
]

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

## RNNs / LSTMs

In [ ]:
def tokenize_text(text):
    return tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=20)

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [torch.tensor(tokenize_text(text)) for text in texts]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

texts, labels = zip(*sarcasm_data)
dataset = SarcasmDataset(list(texts), list(labels))

# Collate function for padding sequences
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    return texts_padded, torch.tensor(labels)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

In [ ]:
class SarcasmLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, output_dim=2):
        super(SarcasmLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

# Model Setup
vocab_size = tokenizer.vocab_size  # Use tokenizer's vocab size
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SarcasmLSTM(vocab_size).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Training Loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 5.4537
Epoch 2, Loss: 4.6590
Epoch 3, Loss: 3.2165
Epoch 4, Loss: 1.1752
Epoch 5, Loss: 0.1740


In [195]:
def predict_sarcasm_RNN(text):
    model.eval()
    inputs = torch.tensor([tokenize_text(text)]).to(device)
    with torch.no_grad():
        logits = model(inputs)
    probs = torch.softmax(logits, dim=1)
    return "Sarcastic" if probs[0][1] > probs[0][0] else "Not Sarcastic"

# Example Predictions
test_sentences = [
    "Oh fantastic, yet another Zoom meeting to sit through.", # Sarcastic
    "I really love this song!",  # Not Sarcastic
    "Oh yes, waking up at 5 AM for work is my absolute favorite part of the day.", # Sarcastic
    "This new movie is just the most entertaining thing I've ever watched."  # Not Sarcastic
]

print("RNNs/LSTMs Predictions")
for sentence in test_sentences:
    print(f"Sentence: {sentence} → Prediction: {predict_sarcasm_RNN(sentence)}")

RNNs/LSTMs Predictions
Sentence: Oh fantastic, yet another Zoom meeting to sit through. → Prediction: Sarcastic
Sentence: I really love this song! → Prediction: Sarcastic
Sentence: Oh yes, waking up at 5 AM for work is my absolute favorite part of the day. → Prediction: Sarcastic
Sentence: This new movie is just the most entertaining thing I've ever watched. → Prediction: Sarcastic


We see that our LSTM is struggling to differentiate between the sarcastic remrks and the non-sarcastic remarks!

* Question: would further training help improve accuracy? 

## Transformers

In [ ]:
class SarcasmDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
        self.encodings = tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors="pt")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, torch.tensor(self.labels[idx])

texts, labels = zip(*sarcasm_data)
dataset = SarcasmDataset(list(texts), list(labels))
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Load Pretrained Model
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Training Setup
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.CrossEntropyLoss()

# Training Loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in dataloader:
        inputs, labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1, Loss: 6.8764
Epoch 2, Loss: 5.7834
Epoch 3, Loss: 4.5483
Epoch 4, Loss: 3.0679
Epoch 5, Loss: 1.3510


In [ ]:
def predict_sarcasm_transformers(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)
    return "Sarcastic" if probs[0][1] > probs[0][0] else "Not Sarcastic"


test_sentences = [
    "Oh fantastic, yet another Zoom meeting to sit through.", # Sarcastic
    "I really love this song!",  # Not Sarcastic
    "Oh yes, waking up at 5 AM for work is my absolute favorite part of the day.", # Sarcastic
    "This new movie is just the most entertaining thing I've ever watched."  # Not Sarcastic
]

print("Transformers (DistilBERT) classifier")

for sentence in test_sentences:
    print(f"Sentence: {sentence}  Prediction: {predict_sarcasm_transformers(sentence)}")

Transformers (DistilBERT) classifiers
Sentence: Oh fantastic, yet another Zoom meeting to sit through.  Prediction: Sarcastic
Sentence: I really love this song!  Prediction: Not Sarcastic
Sentence: Oh yes, waking up at 5 AM for work is my absolute favorite part of the day.  Prediction: Sarcastic
Sentence: This new movie is just the most entertaining thing I've ever watched.  Prediction: Not Sarcastic


Interesting! Our classifier built on the DistilBERT transformer managed to detect the non sarcastic remarks! 

* I wonder what would happen if we were to scale this example up to more rows? 

## 2. Sarcastic News Headlines dataset

* Kaggle link: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection

* Place the dataset file "Sarcasm_Headlines_Dataset_v2.json" in your working directory

In [ ]:
import json
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer

In [254]:
with open("Sarcasm_Headlines_Dataset_v2.json", "r") as file:
    data = [json.loads(line) for line in file]

df = pd.DataFrame(data)
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [281]:
df['headline'][14]

'ford develops new suv that runs purely on gasoline'

In [255]:
df['is_sarcastic'].value_counts()

is_sarcastic
0    14985
1    13634
Name: count, dtype: int64

## Pre-processing

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded_text = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=40,
            return_tensors="pt",
        )
        return encoded_text["input_ids"].squeeze(0), torch.tensor(label, dtype=torch.long)

texts = df["headline"].tolist()
labels = df["is_sarcastic"].tolist()
sarcasm_dataset = SarcasmDataset(texts, labels)

train_loader = DataLoader(sarcasm_dataset, batch_size=16, shuffle=True)

for batch in train_loader:
    input_ids, labels = batch
    print("Sample Input IDs:", input_ids.shape)
    print("Sample Labels:", labels.shape)
    break

Sample Input IDs: torch.Size([16, 40])
Sample Labels: torch.Size([16])


In [241]:
train_loader

## Loading the pre-trained LSTM model

* To save time, you could load the model that has been trained by the code cells below. If you want to train yourself, this only took me 30 seconds per epoch x 5 = 2.5 mins or so. 

Note: below, if you're loading in the pre-trained LSTM model, just be aware that you need to call the constructor with matching parameters (hence they're defined below to match the eval statement).

In [ ]:
class SarcasmLSTM(nn.Module):
    def __init__(self, vocab_size=30522, embedding_dim=128, hidden_dim=128, num_layers=1, output_dim=1):
        super(SarcasmLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # Matches (30522, 128)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)  # Matches (128, 128)
        self.fc = nn.Linear(hidden_dim, output_dim)  # Matches in_features=128, out_features=1
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)  # Convert input IDs to embeddings
        lstm_out, _ = self.lstm(embedded)  # LSTM processing
        final_hidden_state = lstm_out[:, -1, :]  # Take the last hidden state
        output = self.fc(final_hidden_state)  # Fully connected layer
        return self.sigmoid(output)  # Sigmoid for binary classification

In [243]:
lstm_model = SarcasmLSTM()

lstm_model.load_state_dict(torch.load("lstm_sarcasm_model.pth"))

print("LSTM model loaded successfully!")

LSTM model loaded successfully!


/var/folders/ry/3hkntqmd6lx9rvtg9q4zp4vr0000gn/T/ipykernel_74680/1157447422.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lstm_model.load_state_dict(torch.load("lstm_s

In [244]:
lstm_model.eval()

SarcasmLSTM(
  (embedding): Embedding(30522, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

## Training the LSTM model
Or if you want to train the model yourself, be aware that each epoch took me about 30 seconds on my M1 Pro chip, so x 5 epochs should be around 2.5 mins in total.

In [ ]:
class SarcasmLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, output_dim=1):
        super(SarcasmLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        out = self.fc(lstm_out[:, -1, :])
        return self.sigmoid(out).squeeze(1)

VOCAB_SIZE = tokenizer.vocab_size
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "mps" if torch.backends.mps.is_available() else "cpu"

# Initialize model, loss, and optimizer
lstm_model = SarcasmLSTM(VOCAB_SIZE).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Training loop
NUM_EPOCHS = 5
for epoch in range(NUM_EPOCHS):
    lstm_model.train()
    total_loss = 0
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device).float()
        optimizer.zero_grad()
        outputs = lstm_model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, LSTM Loss: {total_loss/len(train_loader):.4f}")

print("LSTM Training Complete!")

Epoch 1, LSTM Loss: 0.6925
Epoch 2, LSTM Loss: 0.6923
Epoch 3, LSTM Loss: 0.5984
Epoch 4, LSTM Loss: 0.3357
Epoch 5, LSTM Loss: 0.2026
LSTM Training Complete!


In [246]:
lstm_model.eval()

SarcasmLSTM(
  (embedding): Embedding(30522, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

## Save the LSTM_model (as a pre-trained model) to save us having to train again in future!

In [248]:
torch.save(lstm_model.state_dict(), "lstm_sarcasm_model.pth")
print("LSTM model saved successfully!")

LSTM model saved successfully!


## Transformers (DistilBERT)

Either load in from pretrained (see the 'distilbert_sarcasm_model.pth'), or if you want to train yourself, just be aware that it took me 3 to 4 mins per epoch on my GPU (M1 Pro chip), so could you take you around 20 mins if you have equivalent hardware.

In [219]:
bert_model = DistilBertForSequenceClassification.from_pretrained("distilbert_sarcasm_model")
bert_model.eval()  # Set to evaluation mode
print("DistilBERT model loaded successfully!")

DistilBERT model loaded successfully!


Or feel free to train yourself (could take up to 20 mins)

In [101]:
from transformers import DistilBertForSequenceClassification, AdamW

# Load pre-trained DistilBERT model
bert_model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)
bert_optimizer = AdamW(bert_model.parameters(), lr=2e-5)

# Training loop for DistilBERT
for epoch in range(NUM_EPOCHS):
    bert_model.train()
    total_loss = 0
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        bert_optimizer.zero_grad()
        outputs = bert_model(input_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        bert_optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, BERT Loss: {total_loss/len(train_loader):.4f}")

print("DistilBERT Training Complete!")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1, BERT Loss: 0.2641
Epoch 2, BERT Loss: 0.1144
Epoch 3, BERT Loss: 0.0449
Epoch 4, BERT Loss: 0.0270
Epoch 5, BERT Loss: 0.0187
DistilBERT Training Complete!


## Save the BERT_model (as a pre-trained model) to save us having to train again in future!

In [ ]:
bert_model.save_pretrained("distilbert_sarcasm_model")
print("DistilBERT model saved successfully!")

DistilBERT model saved successfully!


In [201]:
bert_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## Compare accuracy of the two models

* Took around a minute on my M1 Pro

In [247]:
from sklearn.metrics import accuracy_score
import torch

def evaluate_model(model, dataloader):
    device = next(model.parameters()).device  # Get model's device dynamically
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, labels in dataloader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            outputs = model(input_ids)
            
            # Handle model output format
            if hasattr(outputs, 'logits'):  # Transformer models like DistilBERT
                preds = outputs.logits.argmax(dim=1)
            else:  # LSTM or similar models
                preds = (outputs > 0.5).long()
                
            all_preds.extend(preds.cpu().numpy())  # Move to CPU before converting
            all_labels.extend(labels.cpu().numpy())
    
    return accuracy_score(all_labels, all_preds)

lstm_acc = evaluate_model(lstm_model, train_loader)
bert_acc = evaluate_model(bert_model, train_loader)

print(f"LSTM Accuracy: {lstm_acc:.4f}")
print(f"DistilBERT Accuracy: {bert_acc:.4f}")

LSTM Accuracy: 0.9642
DistilBERT Accuracy: 0.9991


* So they're pretty close - 99% to 92%. 

* LSTMs can learn sarcasm in single-sentence news, but Transformers generalize better.
* Real-world sarcasm is nuanced, so deeper context helps.

* Therefore, perhaps we need an even more nuanced dataset to help illustrate the difference in performance (how well 'long-term dependencies' are captured).

## Your Exercise

Your exercise therefore, is to pick another dataset to exemplify the differences between RNN/LSTM models and Transformer models (e.g. DistilBERT)  
This could be another sarcastic dataset or another type of dataset (e.g. IMBD movies)

* Reddit Sarcasm Dataset (SARC) [Challenging] (for long-range sarcasm and discussions).

* Twitter (X) Irony & Sarcasm Dataset [More Real-World] (for high variation and context-dependency).

* Amazon/Yelp Reviews with Sarcasm Tags (LSTMs may lose long-term dependencies in multi-sentence reviews whereas Transformers track sarcastic tone across the whole review.

* IMDB Movie Classification (for something other than sarcasm - Oh really?! Sorry, couldn't resist...)

## Example: RNNs vs Transformers for Movie Classification 

In [138]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from datasets import load_dataset

In [128]:
# Load IMDb dataset from Hugging Face
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [129]:
dataset.keys()

dict_keys(['train', 'test', 'unsupervised'])

In [112]:
dataset['train']['text'][:10]

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

In [ ]:
# Split into training and testing sets
train_texts = [x["text"] for x in dataset["train"]]
train_labels = [x["label"] for x in dataset["train"]]

test_texts = [x["text"] for x in dataset["test"]]
test_labels = [x["label"] for x in dataset["test"]]

## Or load in the IMDB csv

In [127]:
movies_df = pd.read_csv("IMDB-Movie-Data.csv")
movies_df.sample(5)

,Rank,Title,Genre,Description,Director,Actors,Year,Runtime (Minutes),Rating,Votes,Revenue (Millions),Metascore
521,522,The Counselor,"Crime,Drama,Thriller",A lawyer finds himself in over his head when h...,Ridley Scott,"Michael Fassbender, Penélope Cruz, Cameron Dia...",2013,117,5.3,84927,16.97,48.0
737,738,Body of Lies,"Action,Drama,Romance",A CIA agent on the ground in Jordan hunts down...,Ridley Scott,"Leonardo DiCaprio, Russell Crowe, Mark Strong,...",2008,128,7.1,182305,39.38,57.0
740,741,The Boss,Comedy,A titan of industry is sent to prison after sh...,Ben Falcone,"Melissa McCarthy, Kristen Bell, Peter Dinklage...",2016,99,5.4,29642,63.03,40.0
660,661,Pineapple Express,"Action,Comedy,Crime",A process server and his marijuana dealer wind...,David Gordon Green,"Seth Rogen, James Franco, Gary Cole, Danny McB...",2008,111,7.0,267872,87.34,64.0
411,412,Pitch Perfect 2,"Comedy,Music",After a humiliating command performance at The...,Elizabeth Banks,"Anna Kendrick, Rebel Wilson, Hailee Steinfeld,...",2015,115,6.5,108306,183.44,63.0


## Continue your work here... Try and train both an LSTM and a DistilBERT model then compare their accuracy. 

In [ ]:
## Continue your work here... 